In [1]:
# general and basic imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
from torch_geometric.nn.conv import GraphConv, GINConv, GATv2Conv, TransformerConv
from torch_geometric.utils import from_networkx
from torch_geometric.nn.inits import uniform
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random

C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# running on cpu while the heavier experiments are relegated to the gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# same as in experiment_synthetic_sum
class BaseGNN(nn.Module):
    def __init__(self, init_std=0.01):
        super().__init__()
        self.init_std = init_std

    def init_params(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_normal_(param, gain=self.init_std)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

class StandardGraphConv(BaseGNN):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(GraphConv(in_channels, hidden_channels, aggr='add', bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(GraphConv(hidden_channels, hidden_channels, aggr='add', bias=bias))
        
        self.pool = global_mean_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        
        self.init_params_graphconv()

    def init_params_graphconv(self):
        for conv in self.convs:
            if hasattr(conv, 'lin_root'): nn.init.xavier_normal_(conv.lin_root.weight, gain=self.init_std)
            if hasattr(conv, 'lin_rel'): nn.init.constant_(conv.lin_rel.weight, 0.0) # W2 starts at 0
        if hasattr(self.readout, 'weight'): nn.init.xavier_normal_(self.readout.weight, gain=self.init_std)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [4]:
# same as the StandardGraphConv but with "mean" aggregation
class GraphConvNormalized(BaseGNN):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(GraphConv(in_channels, hidden_channels, aggr='mean', bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(GraphConv(hidden_channels, hidden_channels, aggr='mean', bias=bias))
        
        self.pool = global_mean_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        
        self.init_params_graphconv()

    def init_params_graphconv(self):
        for conv in self.convs:
            if hasattr(conv, 'lin_root'): nn.init.xavier_normal_(conv.lin_root.weight, gain=self.init_std)
            if hasattr(conv, 'lin_rel'): nn.init.constant_(conv.lin_rel.weight, 0.0) # W2 starts at 0
        if hasattr(self.readout, 'weight'): nn.init.xavier_normal_(self.readout.weight, gain=self.init_std)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [5]:
# not adjusted GIN, as simple as possible, just the wrapper
class GINStudent(nn.Module):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=True, **kwargs):
        super().__init__()
        
        self.num_layers = num_layers
        self.convs = nn.ModuleList()

        self.convs.append(GINConv(
            nn.Linear(in_channels, hidden_channels, bias=bias),
            train_eps=True
        ))
        
        for _ in range(num_layers - 1):
            self.convs.append(GINConv(
                nn.Linear(hidden_channels, hidden_channels, bias=bias),
                train_eps=True
            ))

        # using global_add_pool, heavily unstable with global_mean_pool for some reason
        self.pool = global_add_pool
        
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        
        self.init_params_student_gin()

    def init_params_student_gin(self):
        for conv in self.convs:
            uniform(conv.nn.weight.size(0), conv.nn.weight)
            if conv.nn.bias is not None:
                conv.nn.bias.data.zero_()
            if conv.eps is not None:
                conv.eps.data.fill_(0)

    def forward(self, data):
        x = data.x.float()
        edge_index, batch = data.edge_index, data.batch
        
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            
        x = self.pool(x, batch)
        
        return self.readout(x)

In [6]:
# unadjusted GATv2, just the wrapper
class GATv2Model(BaseGNN):
    """GATv2 (Appendix B.2)."""
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=True, heads=1, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(GATv2Conv(in_channels, hidden_channels, heads=heads, concat=False, bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(GATv2Conv(hidden_channels, hidden_channels, heads=heads, concat=False, bias=bias))
            
        self.pool = global_add_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        self.init_params()

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [7]:
# unadjusted GraphTransformer, just the wrapper
class GraphTransformerModel(BaseGNN):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=True, heads=1, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(TransformerConv(in_channels, hidden_channels, heads=heads, concat=False, beta=True, bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, beta=True, bias=bias))
            
        self.pool = global_add_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        self.init_params()

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [8]:
# helper to get the model
def get_model(name, in_channels, out_channels, num_layers, hidden_channels):
    common_args = {
        'in_channels': in_channels,
        'out_channels': out_channels,
        'num_layers': num_layers,
        'hidden_channels': hidden_channels,
        'init_std': 0.01
    }
    
    if name == 'GraphConv':
        return StandardGraphConv(bias=False, **common_args)
    elif name == 'GraphConvNormalized':
        return GraphConvNormalized(bias=False, **common_args)
    elif name == 'GIN':
        return GINStudent(bias=True, **common_args)
    elif name == 'GATv2':
        return GATv2Model(bias=True, heads=4, **common_args)
    elif name == 'GraphTransformer':
        return GraphTransformerModel(bias=True, heads=4, **common_args)
    else:
        raise ValueError(f"Unknown model name: {name}")

In [9]:
# parameters taken from the paper
def generate_dataset(num_samples, graph_type, teacher_weights, num_nodes=20, num_features=16):
    dataset = []
    for _ in range(num_samples):
        if graph_type == 'Empty': G = nx.empty_graph(num_nodes)
        elif graph_type == 'Star': G = nx.star_graph(num_nodes - 1)
        elif graph_type == 'Regular': G = nx.random_regular_graph(d=10, n=num_nodes)
        elif graph_type == 'GNP': G = nx.gnp_random_graph(n=num_nodes, p=0.6)
        elif graph_type == 'BA': G = nx.barabasi_albert_graph(n=num_nodes, m=3)
        
        data = from_networkx(G)
        data.x = torch.randn(num_nodes, num_features)
        graph_sum = data.x.sum(dim=0)
        projection = torch.dot(graph_sum, teacher_weights).item()
        data.y = torch.tensor([1 if projection > 0 else 0], dtype=torch.long)
        dataset.append(data)
    return dataset

In [10]:
import numpy as np

class EarlyStopping:
    def __init__(self, patience=100, delta=0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0  # Reset counter
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print("Early stopping triggered.")

        return self.early_stop

In [11]:
def train_model(model, train_set, val_set, test_set, epochs=200):
    LR = 0.01
    BATCH_SIZE = 8 
    PATIENCE = 100
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, weight_decay = 0.0001)
    criterion = nn.CrossEntropyLoss()
    
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

    early_stopper = EarlyStopping(patience=PATIENCE)
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss_sum = 0
        val_total_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                val_loss_sum += criterion(out, batch.y).item()
                val_total_batches += 1
        
        avg_val_loss = val_loss_sum / val_total_batches

        if early_stopper(avg_val_loss):
            print(f'Early stop triggered at epoch: {epoch}')
            break

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.y.size(0)
            
    return correct / total, model

In [12]:
def plot_accuracy(model, df):   
    plt.figure(figsize=(10, 6))
    ax = plt.gca() 
    
    sns.lineplot(data=df, x='Number of samples', y='Accuracy', hue='Type of graph',
                 style='Type of graph', markers=True, dashes=False, ax=ax, linewidth=2)
    
    ax.set_xscale('log')
    ax.set_xticks(SAMPLE_SIZES)
    ax.set_xticklabels(SAMPLE_SIZES)
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, which="both", ls="-", alpha=0.5)
    ax.set_title("Accuracy vs Samples")
    
    plt.tight_layout()
    plt.savefig(f'{model}_accuracy.jpg', format='jpg', dpi=300)
    
    plt.close()

In [13]:
def plot_weight_ratios(model, df):
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    sns.lineplot(data=df, x='Number of samples', y='Weight Ratio', hue='Type of graph',
                 style='Type of graph', markers=True, dashes=False, ax=ax, linewidth=2)
    
    ax.set_xscale('log')
    ax.set_xticks(SAMPLE_SIZES)
    ax.set_xticklabels(SAMPLE_SIZES)
    ax.set_ylabel(r'$\|W_2\| / \|W_1\|$')
    ax.grid(True, which="both", ls="-", alpha=0.5)
    ax.set_title("Weight Norm Ratio vs Samples")
    
    plt.tight_layout()
    plt.savefig(f'{model}_weight_ratios.jpg', format='jpg', dpi=300)
    plt.close()

In [14]:
# constants and config
NUM_NODES = 20
NUM_FEATURES = 128
SAMPLE_SIZES = [20, 40, 60, 100, 200, 300, 400, 500, 1000, 2000, 4000]
GRAPH_TYPES = ['Empty', 'Regular', 'GNP', 'BA', 'Star']
ALL_MODELS = ['GraphConv', 'GraphConvNormalized', 'GIN', 'GATv2', 'GraphTransformer']
NUM_SEEDS = 3 

HIDDEN_DIM = 64
OUT_CHANNELS = 2
NUM_LAYERS = 1

results = []

print(f"Starting Experiment with D={NUM_FEATURES}...")

teacher_weights = torch.randn(NUM_FEATURES)
teacher_weights = teacher_weights / torch.norm(teacher_weights)

test_sets = {}
val_sets = {}
print("Generating Test Sets...")
for g_type in GRAPH_TYPES:
    test_sets[g_type] = generate_dataset(100, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)
    val_sets[g_type] = generate_dataset(100, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)


for MODEL_NAME in ALL_MODELS:
    print(f"\n======== Running Model: {MODEL_NAME} ========")
    for n_samples in SAMPLE_SIZES:
        print(f"\n--- N = {n_samples} ---")
        
        for g_type in GRAPH_TYPES:
            accuracies = []
            weight_ratios = []
            
            for seed in range(NUM_SEEDS):
                torch.manual_seed(seed + n_samples + 42)
                np.random.seed(seed + n_samples + 42)
                random.seed(seed + n_samples + 42)
                
                train_set = generate_dataset(n_samples, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)

                model = get_model(MODEL_NAME, train_set[0].x.shape[1], OUT_CHANNELS, NUM_LAYERS, HIDDEN_DIM).to(device)
                
                acc, model = train_model(model, train_set, val_sets[g_type], test_sets[g_type], epochs=1000) 
                accuracies.append(acc)
                
                if MODEL_NAME in ['GraphConv', 'GraphConvNormalized']:
                    layer = model.convs[0]
                    
                    w1_norm = torch.norm(layer.lin_root.weight).item()
                    w2_norm = torch.norm(layer.lin_rel.weight).item()
                    
                    if w1_norm > 1e-6:
                        ratio = w2_norm / w1_norm
                    else:
                        ratio = 0.0
                else:
                    ratio = np.nan
                                        
                weight_ratios.append(ratio)
            
            mean_acc = np.mean(accuracies)
            mean_ratio = np.mean(weight_ratios)
            print(f"{g_type}: Acc={mean_acc:.4f}, Ratio={mean_ratio:.4f}")
            
            for i in range(NUM_SEEDS):
                results.append({
                    'Number of samples': n_samples,
                    'Accuracy': accuracies[i],
                    'Weight Ratio': weight_ratios[i],
                    'Type of graph': g_type,
                    'Model': MODEL_NAME 
                })

df = pd.DataFrame(results)
df.to_csv("df_all_models.csv", index=False)
print("\n--- Final results saved to df_all_models.csv ---")

Starting Experiment with D=128...
Generating Test Sets...

======== Running Model: GraphConv ========

--- N = 20 ---
Early stop triggered at epoch: 531
Early stop triggered at epoch: 100
Early stop triggered at epoch: 100
Empty: Acc=0.4767, Ratio=0.0000
Early stop triggered at epoch: 199
Early stop triggered at epoch: 130
Early stop triggered at epoch: 147
Regular: Acc=0.5867, Ratio=6.6755
Early stop triggered at epoch: 147
Early stop triggered at epoch: 162
Early stop triggered at epoch: 145
GNP: Acc=0.5967, Ratio=6.8365
Early stop triggered at epoch: 297
Early stop triggered at epoch: 216
Early stop triggered at epoch: 184
BA: Acc=0.5600, Ratio=4.4790
Early stop triggered at epoch: 222
Early stop triggered at epoch: 222
Early stop triggered at epoch: 221
Star: Acc=0.4900, Ratio=4.2107

--- N = 40 ---
Early stop triggered at epoch: 100
Early stop triggered at epoch: 100
Early stop triggered at epoch: 818
Empty: Acc=0.5200, Ratio=0.0000
Early stop triggered at epoch: 156
Early stop tr

In [15]:
models_to_plot = df['Model'].unique()

for model_name in models_to_plot:
    
    model_df = df[df['Model'] == model_name].copy()
    
    print(f"Generating plots for {model_name}...")
    
    plot_accuracy(model_name, model_df)
    
    if model_name in ['GraphConv', 'GraphConvNormalized']:
        plot_weight_ratios(model_name, model_df)
    else:
        print(f"Skipping weight ratio plot for {model_name}.")

print("All plots generated and saved!")

Generating plots for GraphConv...
Generating plots for GraphConvNormalized...
Generating plots for GIN...
Skipping weight ratio plot for GIN.
Generating plots for GATv2...
Skipping weight ratio plot for GATv2.
Generating plots for GraphTransformer...
Skipping weight ratio plot for GraphTransformer.
All plots generated and saved!
